# thermground: LoRA fine-tuning on Colab

Runs the CUDA part of week 4: a zero-shot baseline with transformers, LoRA fine-tuning and
evaluation of the fine-tuned model. Run the cells top to bottom on a GPU runtime
(L4 or A100).

Everything that should survive a disconnect is written to Google Drive under
`MyDrive/thermground/`: adapters and evaluation outputs. Evaluation
resumes where it stopped, so after a disconnect just run the cells again.

The last cell packs the small result files into `week4_results.zip` for the repository.

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/thermground"
RUNS = f"{DRIVE}/runs"
RESULTS = f"{DRIVE}/results"
!mkdir -p {RUNS} {RESULTS}

In [ ]:
%cd /content
![ -d thermground ] || git clone -q https://github.com/barisparlakk/thermground.git
%cd /content/thermground
!git pull -q && git log --oneline -1
!pip install -q -e ".[train,viz]"

## 2. Data

HIT-UAV is downloaded from Zenodo to the local disk (faster to read than Drive, about
800 MB). The split is read from
`configs/splits/`, and the QA set is regenerated from its config. The checksums must match
the files used on the Mac; otherwise the numbers are not comparable.

In [ ]:
import os

if not os.path.isdir("data/hituav"):
    !python scripts/download_hituav.py
!python scripts/make_hituav_qa.py --stats /content/qa_stats > /dev/null

EXPECTED = {
    "train": "f30c5520a17a53c88da2f26fd445b96e",
    "val": "5206e22835d74a0cfb26ec955f72decf",
    "test": "a7dbbbc15167aa5b9cf09a82b09b6339",
}
import hashlib

for split, md5 in EXPECTED.items():
    got = hashlib.md5(open(f"data/qa/hituav_qa_v1/{split}.jsonl", "rb").read()).hexdigest()
    assert got == md5, f"{split}.jsonl differs from the Mac version"
print("QA files match")

## 3. Pipeline check (2B, a few minutes)

Five training steps and a 20-item evaluation, to catch problems before the long run.

In [ ]:
!python scripts/train_lora.py --config configs/train/qwen3_vl_2b_lora_v1.yaml --out /content/smoke --max-steps 5
!python scripts/evaluate.py --model /content/smoke/model.yaml --set val-mini --limit 20 --conditions real --out /content/smoke/eval

## 4. Zero-shot baseline with transformers (4B)

The Mac numbers come from a 4-bit MLX build, so the fine-tuned model is compared with the
same bf16 model, zero-shot, on the same backend.

In [ ]:
!python scripts/evaluate.py --model configs/models/qwen3_vl_4b_hf.yaml --set val-mini --out {RESULTS}/week4/eval/qwen3_vl_4b_hf/val-mini

## 5. LoRA fine-tuning (4B)

In [ ]:
!python scripts/train_lora.py --config configs/train/qwen3_vl_4b_lora_v1.yaml --out {RUNS}/qwen3_vl_4b_lora_v1

## 6. Evaluate the fine-tuned model

val-mini first (decisions); test-mini is the weekly reported number.

In [ ]:
MODEL = f"{RUNS}/qwen3_vl_4b_lora_v1/model.yaml"
for s in ["val-mini", "test-mini"]:
    !python scripts/evaluate.py --model {MODEL} --set {s} --out {RESULTS}/week4/eval/qwen3_vl_4b_lora_v1/{s}

In [ ]:
!python scripts/evaluate.py --model configs/models/qwen3_vl_4b_hf.yaml --set test-mini --out {RESULTS}/week4/eval/qwen3_vl_4b_hf/test-mini

## 7. Pack the results

Only small files: metrics, summaries, training logs and configs. Predictions and adapters
stay on Drive. Unzip `week4_results.zip` at the repository root.

In [ ]:
import shutil
from pathlib import Path

pack = Path("/content/week4_results")
shutil.rmtree(pack, ignore_errors=True)
for f in Path(RESULTS).glob("week4/eval/*/*/*"):
    if f.name in ("metrics.json", "summary.md"):
        dest = pack / "results" / f.relative_to(RESULTS)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(f, dest)
for run in Path(RUNS).glob("*"):
    for name in ("train_log.jsonl", "train_config.yaml"):
        if (run / name).exists():
            dest = pack / "results" / "week4" / "train" / run.name / name
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy(run / name, dest)
shutil.make_archive("/content/week4_results", "zip", pack)
!cd {pack} && find . -type f | sort

from google.colab import files

files.download("/content/week4_results.zip")